# CompBench Metrics (Mixed Prompts)

Scores saved mixed-prompt outputs; this notebook does not run the editing agent. It selects the latest CompBench annotation and inference runs by default. Set `MMEDIT_COMPBENCH_ANNOTATION_RUN` or `MMEDIT_COMPBENCH_MIXED_RUN` to select a specific `run_<timestamp>`. Per-sample scores and task/overall means are saved under the selected inference run's `metrics/` directory.

This reproduces the five automated metrics in the [CompBench paper](https://arxiv.org/abs/2505.12200) and [released evaluator](https://github.com/comp-bench/CompBench): LC-T (local CLIP text), LC-I (local CLIP image similarity to the edited ground truth), and background PSNR, SSIM, and LPIPS. Images and masks are resized to 512×512 (bicubic for RGB, nearest-neighbor for masks); LC-T/LC-I use the edit-mask bounding-box crop, while background metrics compare source and generated images with the edited region masked out. Multi-object caption scores are averaged across the pipe-separated captions. The paper also reports separate 0–10 VLM/human judgments for action, location, and viewpoint tasks; those judgments are not generated by these automated metrics.


CompBench annotation files can be sparse: `metadata.row_index` is the original dataset row, not the record position. Saved outputs are matched by `sample_id`.

In [5]:
import os

from evaluation.notebook import load_benchmark, resolve_run_directory

BENCHMARK = "compbench"
PROMPT_MODE = "mixed"
context = load_benchmark(BENCHMARK, prompt_mode=PROMPT_MODE)
ANNOTATION_FILE = context.annotation_file
OUTPUT_ROOT = context.output_dir
RUN_NAME = os.environ.get("MMEDIT_COMPBENCH_MIXED_RUN") or None
RUN_NAME = "run_20261011_214946_086189"
OUTPUT_DIR = resolve_run_directory(OUTPUT_ROOT, RUN_NAME)
records = context.records
image_loader = context.image_loader
CLIP_MODEL_NAME = os.environ.get(
    "COMPBENCH_CLIP_MODEL", "openai/clip-vit-large-patch14"
)
DEVICE = os.environ.get("COMPBENCH_METRICS_DEVICE") or None
print("Annotations:", ANNOTATION_FILE)
print("Selected run:", OUTPUT_DIR)
print("CLIP model:", CLIP_MODEL_NAME)
print("Metrics device:", DEVICE or "auto")

Annotations: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/annotation/compbench/run_20261007_000000_000000/compbench_local_multiobject_mixed.jsonl
Selected run: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/evaluation/compbench/mixed/run_20261011_214946_086189
CLIP model: openai/clip-vit-large-patch14
Metrics device: auto


In [6]:
from evaluation.compbench_metrics import collect_saved_outputs

saved_outputs = collect_saved_outputs(
    records, OUTPUT_DIR, prompt_mode=PROMPT_MODE
)
print(f"Found {len(saved_outputs)} saved output images:")
for saved in saved_outputs:
    print(f"  [{saved['record_position']}] {saved['sample_id']}")

Found 2609 saved output images:
  [0] add:001ca3cb/0/00003.png
  [1] add:001ca3cb/0/00005.png
  [2] add:001ca3cb/0/00025.png
  [3] add:001ca3cb/0/00027.png
  [4] add:02221fb0/1/00001.png
  [5] add:02221fb0/1/00003.png
  [6] add:02221fb0/1/00014.png
  [7] add:02221fb0/1/00023.png
  [8] add:02221fb0/1/00038.png
  [9] add:02b72b8d/0/00066.png
  [10] add:02b72b8d/1/00013.png
  [11] add:02b72b8d/1/00017.png
  [12] add:02b72b8d/2/00016.png
  [13] add:02b72b8d/2/00052.png
  [14] add:02b72b8d/3/00003.png
  [15] add:02b72b8d/3/00015.png
  [16] add:02b72b8d/3/00019.png
  [17] add:02b72b8d/4/00003.png
  [18] add:03012cfd/0/00083.png
  [19] add:03c21af7/0/00018.png
  [20] add:03c21af7/0/00021.png
  [21] add:03c21af7/1/00020.png
  [22] add:03c21af7/3/00020.png
  [23] add:03c21af7/3/00026.png
  [24] add:03c21af7/3/00030.png
  [25] add:03c21af7/3/00032.png
  [26] add:03c21af7/3/00035.png
  [27] add:03e5f069/1/00086.png
  [28] add:05ed1023/0/00006.png
  [29] add:05ed1023/0/00010.png
  [30] add:0b27d5f

## Score saved outputs

Install the optional metrics dependencies with `uv sync --extra metrics` in the project environment if needed, then select that environment as the notebook kernel. The first run may download the CLIP model. Scores are cached by output path, size, and modification time; progress is checkpointed every 25 samples.

This cell runs CompBench's five standard metrics across the selected annotation set. The separate VIEScore cell below is restricted to exactly `multi_object_add` and `multi_object_remove`.

In [2]:
from IPython.display import display

from evaluation.compbench_metrics import evaluate_saved_compbench_outputs

metric_rows, score_summary = evaluate_saved_compbench_outputs(
    records,
    output_dir=OUTPUT_DIR,
    image_loader=image_loader,
    prompt_mode=PROMPT_MODE,
    clip_model_name=CLIP_MODEL_NAME,
    device_name=DEVICE,
)
display(
    [
        {"task": task, **metrics}
        for task, metrics in score_summary.items()
    ]
)

FileNotFoundError: No saved output images found in /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/evaluation/compbench/mixed/run_20261018_020007_990052/results/images

In [ ]:
from IPython.display import Markdown, display

CATEGORY_TASKS = {
    "Local": ("add", "remove", "replace"),
    "Multi-object": ("multi_object_add", "multi_object_remove"),
}
METRICS = ("SSIM", "PSNR", "LPIPS", "LC-T", "LC-I")


def weighted_category_summary(tasks):
    task_summaries = [score_summary[task] for task in tasks if task in score_summary]
    if len(task_summaries) != len(tasks):
        missing = sorted(set(tasks) - set(score_summary))
        raise ValueError(f"Missing task summaries: {missing}")
    count = sum(summary["count"] for summary in task_summaries)
    return {
        "count": count,
        **{
            metric: sum(summary[metric] * summary["count"] for summary in task_summaries)
            / count
            for metric in METRICS
        },
    }


local = weighted_category_summary(CATEGORY_TASKS["Local"])
multi_object = weighted_category_summary(CATEGORY_TASKS["Multi-object"])
columns = [
    "Model",
    "Local n",
    "Local SSIM ↑",
    "Local PSNR ↑",
    "Local LPIPS ↓",
    "Local LC-T ↑",
    "Local LC-I ↑",
    "Multi-object n",
    "Multi-object SSIM ↑",
    "Multi-object PSNR ↑",
    "Multi-object LPIPS ↓",
    "Multi-object LC-T ↑",
    "Multi-object LC-I ↑",
]
values = [
    f"MMEdit ({PROMPT_MODE})",
    str(local["count"]),
    *(f"{local[metric]:.4f}" for metric in METRICS),
    str(multi_object["count"]),
    *(f"{multi_object[metric]:.4f}" for metric in METRICS),
]
markdown_table = "| " + " | ".join(columns) + " |\n"
markdown_table += "| " + " | ".join(["---"] * len(columns)) + " |\n"
markdown_table += "| " + " | ".join(values) + " |"
display(Markdown(f"## CompBench table ({PROMPT_MODE})\n\n" + markdown_table))

## VIEScore (multi-object tasks only)

Scores only records whose task is exactly `multi_object_add` or `multi_object_remove` (not the single-object `add`/`remove` tasks or `replace`). It uses the original instruction, the dataset edit mask, and matching tight crops of the source and generated images. This runs the GEdit-Bench VIEScore `tie` task with GPT-4o. API usage may incur charges. Set `VIESCORE_ROOT` to the local GEdit-Bench directory and `VIESCORE_KEY_PATH` to a local file containing your OpenAI API key on its first line. Never put the key in a cell or commit its file.

In [3]:
from pathlib import Path
from IPython.display import display

from evaluation.compbench_viescore import evaluate_saved_compbench_outputs

VIESCORE_ROOT = Path(os.environ.get("VIESCORE_ROOT") or ".").expanduser()
VIESCORE_KEY_PATH = Path(os.environ.get("VIESCORE_KEY_PATH") or ".").expanduser()
viescore_rows, viescore_summary = evaluate_saved_compbench_outputs(
    records,
    output_dir=OUTPUT_DIR,
    image_loader=image_loader,
    prompt_mode=PROMPT_MODE,
    viescore_root=VIESCORE_ROOT,
    key_path=VIESCORE_KEY_PATH,
)
print(f"Per-sample VIEScore: {OUTPUT_DIR / 'metrics' / 'modified_viescore.jsonl'}")
display({"overall": viescore_summary.get("overall"), "by_task": viescore_summary["by_task"]})

CompBench VIEScore: 100%|██████████| 144/144 [09:21<00:00,  3.90s/sample]

Scored 144 saved CompBench multi-object outputs.
Per-sample VIEScore: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/evaluation/compbench/mixed/run_20261011_214946_086189/metrics/modified_viescore.jsonl
Task summary: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/evaluation/compbench/mixed/run_20261011_214946_086189/metrics/modified_viescore_summary.json
Per-sample VIEScore: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/evaluation/compbench/mixed/run_20261011_214946_086189/metrics/modified_viescore.jsonl


{'overall': {'sc': 7.347222222222222,
  'pq': 3.7152777777777777,
  'o': 4.902055591462327,
  'count': 144},
 'by_task': {'multi_object_add': {'sc': 7.861111111111111,
   'pq': 4.597222222222222,
   'o': 5.733863099555102,
   'count': 72},
  'multi_object_remove': {'sc': 6.833333333333333,
   'pq': 2.8333333333333335,
   'o': 4.070248083369552,
   'count': 72}}}

## EditScore (multi-object tasks only)

Scores the same `multi_object_add` and `multi_object_remove` outputs as VIEScore, using the original full instruction and full source/generated images. Install the official scorer with `pip install -U editscore` in the notebook environment. The default is EditScore's Qwen3-VL 4B model and LoRA; the weights download on first use. On Apple Silicon, MPS is selected automatically when CUDA is unavailable and the model runs in float16; set `EDITSCORE_DEVICE=mps` to request MPS explicitly. The model still needs enough unified memory. Set `EDITSCORE_BACKBONE`, `EDITSCORE_MODEL`, `EDITSCORE_LORA`, `EDITSCORE_NUM_PASS`, or `EDITSCORE_SCORE_RANGE` to override the defaults. Scores are cached in `editscore.jsonl` and summarized under the selected run's `metrics/` directory.

In [8]:
from evaluation.compbench_editscore import evaluate_saved_compbench_outputs as evaluate_compbench_editscore

editscore_rows, editscore_summary = evaluate_compbench_editscore(
    records,
    output_dir=OUTPUT_DIR,
    image_loader=image_loader,
    prompt_mode=PROMPT_MODE,
)
display(editscore_summary)

Using EditScore qwen3vl on Apple MPS (float16).


CompBench EditScore: 100%|██████████| 144/144 [15:38<00:00,  6.51s/sample]

Per-sample EditScore: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/evaluation/compbench/mixed/run_20261011_214946_086189/metrics/editscore.jsonl
EditScore summary: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/evaluation/compbench/mixed/run_20261011_214946_086189/metrics/editscore_summary.json


{'overall': {'prompt_following': 9.161111111111111,
  'consistency': 9.11111111111111,
  'perceptual_quality': 7.0888888888888895,
  'overall': 7.6965313133554485,
  'count': 144},
 'by_task': {'multi_object_add': {'prompt_following': 9.61111111111111,
   'consistency': 9.333333333333334,
   'perceptual_quality': 6.777777777777778,
   'overall': 7.656497986208682,
   'count': 72},
  'multi_object_remove': {'prompt_following': 8.711111111111112,
   'consistency': 8.88888888888889,
   'perceptual_quality': 7.400000000000001,
   'overall': 7.736564640502215,
   'count': 72}}}